# Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency: A Clinical Analysis and Literature Review Exploration with `mlcroissant`
This notebook provides step-by-step guidance for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset follows the Croissant schema and is available at:

`https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json`


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print(f"Identifier: {metadata.identifier}")
print(f"Published: {metadata.datePublished}\n")
print("Keywords:", metadata.keywords)
print("Use cases:", metadata.dataUseCases)


## 2. Data Overview
Review available record sets, fields, and their `@id`s. For this dataset, each table is represented as a separate record set in the Croissant schema.

Record sets are structured tables. We will fetch the available record sets and summarize their fields by `@id`.

In [ ]:
# List available record sets (tables) and their fields
record_sets = dataset.record_sets
print(f"Dataset contains {len(record_sets)} record sets.")

record_sets_ids = []
for rs in record_sets:
    print(f"RecordSet name: {rs.name}; @id: {rs.id}")
    record_sets_ids.append(rs.id)

    # List fields (columns) for this record set
    print("Fields in RecordSet:")
    for field in rs.fields:
        print(f"  - field name: {field.name}; @id: {field.id}; type: {field.data_type}")
    print()


## 3. Data Extraction
Load data from a specific record set (@id - which corresponds to each table) into a DataFrame for analysis. Use the record set and field `@id`s identified above.

In [ ]:
# Extract data from all available record sets into pandas DataFrames
dataframes = {}

for record_set_id in record_sets_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)

# Show columns for the first record set and preview
first_rs = record_sets_ids[0]
print(f"Columns for RecordSet @id: {first_rs}")
print(dataframes[first_rs].columns.tolist())
dataframes[first_rs].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

The dataset divides clinical, hormone, and genetic info into different tables. Let's pick a numeric field from hormone measurements for demonstration (for example, "serum_17OH_progesterone") and group by phenotype or genotype in the genetic record set if such columns exist.

In [ ]:
# Example: EDA on the hormone record set
# Replace with exact column @id as listed in the overview
hormone_rs = None
hormone_numeric_field = None

# Find hormone record set and a numeric field within it
for rs in record_sets:
    for field in rs.fields:
        # Use known field types or names to select hormone table and column
        if 'hormone' in rs.name.lower():
            hormone_rs = rs.id
            # Select numeric column, e.g., 17OH Progesterone, Testosterone, etc.
            if field.data_type in ['Float', 'Integer']:
                hormone_numeric_field = field.id
                break
    if hormone_rs and hormone_numeric_field:
        break

if hormone_rs and hormone_numeric_field:
    print(f"Using RecordSet: {hormone_rs} and numeric field: {hormone_numeric_field}")
    # Set threshold arbitrarily for the example.
    threshold = 10
    df = dataframes[hormone_rs]
    filtered_df = df[df[hormone_numeric_field] > threshold]
    print(f"Filtered records with {hormone_numeric_field} > {threshold}:")
    print(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{hormone_numeric_field}_normalized"] = (
        filtered_df[hormone_numeric_field] - filtered_df[hormone_numeric_field].mean()
    ) / filtered_df[hormone_numeric_field].std()
    print(f"Normalized {hormone_numeric_field} for filtered records:")
    print(filtered_df[[hormone_numeric_field, f"{hormone_numeric_field}_normalized"]].head())

    # Try grouping by another field if present, e.g., 'phenotype' or 'patient_id'
    group_field_id = None
    for col in df.columns:
        if 'phenotype' in col.lower() or 'patient' in col.lower():
            group_field_id = col
            break
    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"Grouped data by {group_field_id}:")
        print(grouped_df.head())
else:
    print("Could not automatically identify hormone record set or numeric field. Please refer to the overview above and manually select the desired RecordSet @id and field @id.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

We'll visualize the normalized hormone values for each patient or group.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if hormone_rs and hormone_numeric_field and not filtered_df.empty:
    plt.figure(figsize=(8, 5))
    # Plot normalized values
    if group_field_id:
        sns.barplot(data=filtered_df, x=group_field_id, y=f"{hormone_numeric_field}_normalized")
        plt.title(f"Normalized {hormone_numeric_field} by {group_field_id}")
    else:
        filtered_df[f"index"] = filtered_df.index
        sns.barplot(data=filtered_df, x="index", y=f"{hormone_numeric_field}_normalized")
        plt.title(f"Normalized {hormone_numeric_field} per record")
    plt.ylabel(f"Normalized {hormone_numeric_field}")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
else:
    print("No hormone data or numeric field available for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The dataset provides structured tables on clinical, hormone, and genetic features for three female patients with non-classical 21-hydroxylase deficiency.
- Using `mlcroissant`, metadata and table schemas are explored programmatically, referencing entities by their `@id`.
- Numeric hormone measurements can be filtered and normalized, with potential for grouping by phenotype or genotype, illustrating heterogeneity across patients.
- The small sample size limits inference/generalizability, but enables case-based exploration and reproducibility.

Further analyses can be performed by referencing and extracting additional fields and tables via their `@id` for deeper clinical or genetic investigation.